# 并行实验：成对排序 (Pairwise Ranking)

**评估指标说明：**
根据论文公式 (31)，本实验的核心评估指标是 **系数的 RMSE（即 L2 范数距离）**：
$$ \text{RMSE} = \frac{1}{S} \sum_{s=1}^S \sqrt{\sum_{l=1}^p (\hat{\theta}_{l,s} - \theta_l^*)^2} $$
我们在代码中严格遵循了这一指标。此外，我们保留了“成对相关性 (Pairwise Correlation)”作为辅助业务指标。

In [1]:
import os
import json
import time
import numpy as np
%load_ext autoreload
%autoreload 2
import matplotlib.pyplot as plt
from multiprocessing import Pool
from utils.runner import run_single_ranking


In [ ]:
import os
import json  # 必须导入，用于保存文件
import numpy as np
from joblib import Parallel, delayed
from utils.runner import run_single_ranking, run_single_aft

# ==========================================
# 参数设定 (Hyperparameters)
# ==========================================
NUM_RUNS = 200       
NUM_WORKERS = 10     
params = {
    'Experiment': 'Pairwise Ranking',
    'm': 10,
    'n': 100,  # ⚠️ U-统计量 O(n^2)，调参建议 n=100
    'p_prime': 5,
    'p': 50,
    'pc': 0.3,
    'T': 50,
    'W_inner': 5,
    'rho': 1,
    'ic_type': 'bic',
    'lambda_candidates': np.logspace(-2, -1, 10),
    'noise_type': 't1',
    'noise_scale': 0.1,
    "run_proposed": True, "run_local": True, "run_baselines": True
}

# 路径管理
folder = "ranking"
os.makedirs(folder, exist_ok=True)
filename = f"{folder}/exp_{folder}_{NUM_RUNS}_{params['noise_type']}_p{params['p']}_pc_{str(params['pc']).replace('.', '')}_rho_{str(params['rho']).replace('.', '')}.json"

# 选择执行器
runner_fn = run_single_ranking if "Ranking" in params["Experiment"] else run_single_aft

print(f"Starting Parallel Experiments: {NUM_RUNS} runs...")

# 【关键点】加入 verbose 参数
# verbose=10 会实时在下方打印日志，显示已完成的比例和预计剩余时间
results = Parallel(n_jobs=NUM_WORKERS, verbose=10)(
    delayed(runner_fn)(i, params) for i in range(NUM_RUNS)
)

# 保存数据
with open(filename, 'w', encoding='utf-8') as f:
    json.dump({'parameters': params, 'results': results}, f, ensure_ascii=False, indent=4)

print(f"\nCompleted! Results saved to {filename}")

Starting Parallel Experiments: 200 runs...


[Parallel(n_jobs=10)]: Using backend LokyBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done   5 tasks      | elapsed: 38.8min
[Parallel(n_jobs=10)]: Done  12 tasks      | elapsed: 77.2min
[Parallel(n_jobs=10)]: Done  21 tasks      | elapsed: 115.7min
[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed: 117.7min
[Parallel(n_jobs=10)]: Done  41 tasks      | elapsed: 192.1min
[Parallel(n_jobs=10)]: Done  52 tasks      | elapsed: 230.3min
[Parallel(n_jobs=10)]: Done  65 tasks      | elapsed: 269.6min
[Parallel(n_jobs=10)]: Done  78 tasks      | elapsed: 309.2min
[Parallel(n_jobs=10)]: Done  93 tasks      | elapsed: 385.1min
[Parallel(n_jobs=10)]: Done 108 tasks      | elapsed: 426.1min
[Parallel(n_jobs=10)]: Done 125 tasks      | elapsed: 500.6min
[Parallel(n_jobs=10)]: Done 142 tasks      | elapsed: 573.9min


In [ ]:
# ==========================================
# 实验完成
# ==========================================
print(f"\n实验完成！数据已保存至 {filename}")
print("请打开 exp1_plot_ranking.ipynb 读取该文件以生成 Markdown 表格和收敛曲线。\n")
